In [4]:
# ==========================================================
# FEDERATED PERCEPTRON (ONLINE ONLY, NO OFFLINE)
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
LR = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# DATA LOADING
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(np.int64)
    return X, y

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

# ==========================================================
# LOAD CLIENT DATA (NO SPLIT)
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_train_loaders = []
client_test_loaders = []
client_test_labels = []

for f in train_files:
    X_train, y_train = load_csv(os.path.join(DATA_DIR, f))
    X_test, y_test = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))

    client_train_loaders.append(make_loader(X_train, y_train, shuffle=True))
    client_test_loaders.append(make_loader(X_test, y_test, shuffle=False))
    client_test_labels.append(y_test)

# Global test
X_global, y_global = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(X_global, y_global, shuffle=False)

input_dim = X_global.shape[1]
num_clients = len(client_train_loaders)

# ==========================================================
# PERCEPTRON MODEL
# ==========================================================
class Perceptron(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)

# ==========================================================
# EVALUATION
# ==========================================================
def eval_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for Xb, _ in loader:
            Xb = Xb.to(DEVICE)
            p = torch.sigmoid(model(Xb))
            preds.extend((p >= 0.5).cpu().numpy())
    return np.array(preds)

def acc(p, y):
    return np.mean(p == y) * 100

# ==========================================================
# ONLINE TRAINING (STREAMING FULL DATA)
# ==========================================================
def train_stream(model, loader):
    model.train()
    opt = optim.SGD(model.parameters(), lr=LR)
    crit = nn.BCEWithLogitsLoss()

    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE).float()

        opt.zero_grad()
        loss = crit(model(Xb), yb)
        loss.backward()
        opt.step()

# ==========================================================
# INIT CLIENT MODELS
# ==========================================================
global_model = Perceptron(input_dim).to(DEVICE)

# ==========================================================
# FEDERATED ONLINE LOOP
# ==========================================================
for rnd in range(1, FL_ROUNDS + 1):
    print(f"\n========== ROUND {rnd} ==========")

    client_models = []

    # -------------------
    # CLIENT TRAINING
    # -------------------
    for c in range(num_clients):
        model = Perceptron(input_dim).to(DEVICE)
        model.load_state_dict(global_model.state_dict())

        # ONLINE TRAINING ON FULL DATA STREAM
        train_stream(model, client_train_loaders[c])

        client_models.append(model)

        # LOCAL TEST
        preds_local = eval_model(model, client_test_loaders[c])
        local_acc = acc(preds_local, client_test_labels[c])

        print(f"Client {c} Local Test Acc: {local_acc:.2f}%")

    # -------------------
    # AVG LOCAL ACC
    # -------------------
    local_accs = []
    for c in range(num_clients):
        preds_local = eval_model(client_models[c], client_test_loaders[c])
        local_accs.append(acc(preds_local, client_test_labels[c]))

    print(f"Avg Local Test Acc: {np.mean(local_accs):.2f}%")

    # -------------------
    # GLOBAL AGGREGATION
    # -------------------
    agg = {k: torch.zeros_like(v) for k, v in global_model.state_dict().items()}

    for m in client_models:
        for k in agg:
            agg[k] += m.state_dict()[k]

    for k in agg:
        agg[k] /= num_clients

    global_model.load_state_dict(agg)

    # -------------------
    # GLOBAL TEST (PER CLIENT MODEL)
    # -------------------
    global_preds = eval_model(global_model, global_loader)
    global_acc = acc(global_preds, y_global)

    print(f"Global Test Accuracy: {global_acc:.2f}%")

# ==========================================================
# FINAL GLOBAL EVAL
# ==========================================================
print("\n===== FINAL GLOBAL PERFORMANCE =====")
final_preds = eval_model(global_model, global_loader)
final_acc = acc(final_preds, y_global)
print(f"Final Global Test Accuracy: {final_acc:.2f}%")


========== ROUND 1 ==========
Client 0 Local Test Acc: 82.21%
Client 1 Local Test Acc: 94.37%
Client 2 Local Test Acc: 87.10%
Client 3 Local Test Acc: 65.09%
Client 4 Local Test Acc: 71.34%
Client 5 Local Test Acc: 65.98%
Avg Local Test Acc: 77.68%
Global Test Accuracy: 71.80%

========== ROUND 2 ==========
Client 0 Local Test Acc: 82.31%
Client 1 Local Test Acc: 94.38%
Client 2 Local Test Acc: 87.91%
Client 3 Local Test Acc: 66.06%
Client 4 Local Test Acc: 71.62%
Client 5 Local Test Acc: 66.17%
Avg Local Test Acc: 78.08%
Global Test Accuracy: 72.67%

========== ROUND 3 ==========
Client 0 Local Test Acc: 82.31%
Client 1 Local Test Acc: 94.35%
Client 2 Local Test Acc: 88.78%
Client 3 Local Test Acc: 66.35%
Client 4 Local Test Acc: 71.82%
Client 5 Local Test Acc: 66.23%
Avg Local Test Acc: 78.31%
Global Test Accuracy: 73.70%

========== ROUND 4 ==========
Client 0 Local Test Acc: 82.39%
Client 1 Local Test Acc: 94.51%
Client 2 Local Test Acc: 91.13%
Client 3 Local Test Acc: 66.64%
Clie